# Day 10: Build a football star schema

**Phase 1: Foundations & Infrastructure**

## Objective
Build a football star schema

## Task
- Designed a Star Schema architecture, documenting the relationships between facts and dimensions using a Mermaid Entity-Relationship Diagram (ERD).
- Refactored `src/data/loaders.py` to upgrade from `.csv` to optimized `.parquet` format via `pyarrow`, significantly improving I/O performance.
- Separated StatsBomb data extraction into `load_statsbomb_matches` and `load_statsbomb_events` to properly feed dimension and fact tables.
- Developed an ETL pipeline in Pandas to extract raw data and transform it into structured tables (`dim_matches`, `dim_players`, `dim_teams`, `fact_events`).
- Standardized complex data structures by unpacking nested location lists into discrete numerical columns (`location_x`, `location_y`) and casting dictionaries to strings to prevent Parquet serialization errors.
- Loaded the analytical database into the `data/processed/football/` directory, ready for downstream ML applications.

**Deliverable:** ERD diagram + transformation script + processed data in data/processed/football/.

---

### Instructions:
*Treat this as your course workbook. Write your code, notes, or execution steps below.*

Remember: 
- Document your decisions.
- Use clear variable names.
- If today's task involves creating a script in `src/`, a dashboard in `apps/`, or documentation in `docs/`, you can use this notebook to test your logic or simply write your reflections and proofs of execution (e.g. running terminal commands with `!`).



### (Star Schema) - StatsBomb

```mermaid
erDiagram
    fact_events {
        string event_id PK
        string match_id FK
        string player_id FK
        string team_id FK
        int date_id FK
        float location_x
        float location_y
        float pass_length
        float pass_angle
        float expected_goals
    }
    
    dim_players {
        string player_id PK
        string player_name
        string country
        string primary_position
    }
    
    dim_teams {
        string team_id PK
        string team_name
    }
    
    dim_matches {
        string match_id PK
        string season_id FK
        string stadium_id FK
        string referee_id FK
        int home_score
        int away_score
    }
    
    dim_date {
        int date_id PK
        date full_date
        int year
        int month
        string day_of_week
    }
    
    dim_stadiums {
        string stadium_id PK
        string stadium_name
        string country
    }

    dim_players ||--o{ fact_events : "realiza"
    dim_teams ||--o{ fact_events : "pertenece_a"
    dim_matches ||--o{ fact_events : "contiene"
    dim_date ||--o{ fact_events : "ocurre_en"

In [ ]:
import sys
import os
import pandas as pd

# 1. Path configuration and imports
sys.path.append(os.path.abspath('../../')) 
from src.data import loaders

PROCESSED_DIR = "../../data/processed/football/"
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("--- Starting ETL Pipeline (Star Schema) ---")

# --- EXTRACT ---
print("\n1. Extracting data...")
# Download the matches and then the thousands of events from the 2022 World Cup
matches_df = loaders.load_statsbomb_matches(competition_id=43, season_id=106)
events_df = loaders.load_statsbomb_events(competition_id=43, season_id=106)

# --- TRANSFORM ---
print("\n2. Transforming dimensions and facts...")

# A. Matches Dimension
dim_matches = matches_df[['match_id', 'match_date', 'competition_stage', 'stadium', 'referee']].copy()
# StatsBomb sometimes nests dictionaries. Convert to string to avoid Parquet errors
for col in ['stadium', 'referee', 'competition_stage']:
    if col in dim_matches.columns:
        dim_matches[col] = dim_matches[col].astype(str)

# B. Players and Teams Dimensions
# StatsBomb uses 'player' and 'team', rename them to follow our convention
dim_players = events_df[['player_id', 'player']].dropna(subset=['player_id']).drop_duplicates(subset=['player_id'])
dim_players.rename(columns={'player': 'player_name'}, inplace=True)

dim_teams = events_df[['team_id', 'team']].dropna(subset=['team_id']).drop_duplicates(subset=['team_id'])
dim_teams.rename(columns={'team': 'team_name'}, inplace=True)

# C. Fact Table (Events)
fact_columns = [
    'id', 'match_id', 'player_id', 'team_id', 'type', 
    'minute', 'second', 'location', 'pass_length', 'pass_angle', 'shot_statsbomb_xg'
]
existing_cols = [col for col in fact_columns if col in events_df.columns]
fact_events = events_df[existing_cols].copy()

if 'id' in fact_events.columns:
    fact_events.rename(columns={'id': 'event_id'}, inplace=True)

# Crucial transformation: StatsBomb provides 'location' as a list [x, y]. 
# We need to split it into two flat numerical columns.
if 'location' in fact_events.columns:
    # Filter out nulls before expanding the list
    loc_mask = fact_events['location'].notna()
    fact_events.loc[loc_mask, ['location_x', 'location_y']] = pd.DataFrame(
        fact_events.loc[loc_mask, 'location'].tolist(), 
        index=fact_events.loc[loc_mask].index
    )
    fact_events.drop(columns=['location'], inplace=True)

# --- LOAD ---
print("\n3. Loading to Parquet format...")
dim_matches.to_parquet(f"{PROCESSED_DIR}dim_matches.parquet", index=False)
dim_players.to_parquet(f"{PROCESSED_DIR}dim_players.parquet", index=False)
dim_teams.to_parquet(f"{PROCESSED_DIR}dim_teams.parquet", index=False)
fact_events.to_parquet(f"{PROCESSED_DIR}fact_events.parquet", index=False)

print(f"Pipeline complete! Your analytical database is ready at: {PROCESSED_DIR}")